In [1]:
import numpy as np
import os
import torch
import torch.nn as nn
import copy
import pandas as pd
from metabci.brainda.algorithms.deep_learning import EEGNet
from metabci.brainda.algorithms.utils.model_selection import (
    set_random_seeds,
    generate_kfold_indices, match_kfold_indices)
from metabci.brainda.datasets import Wang2016, BETA

from metabci.brainda.paradigms import SSVEP
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EpochScoring, EarlyStopping, EarlyStopping

In [2]:
#**************************************************
# BETA数据集读取处理
#**************************************************
# BETA数据集，已经通过matlab的eegfit，进行了3-90HZ的带通滤波，故此处不再进行滤波处理
# BETA_wof数据集没有进行滤波处理，链接已经
# 在tsinghua.py把BETA_URL做以下修改即可使用未滤波的版本
# BETA_URL = "https://bci.med.tsinghua.edu.cn/upload/liubingchuan_BETA_wof/"


# 数据存储为一个四维张量 [channel, time point, block, condition]
#                    [   64,       750,      4,      40    ]

BETA_dataset = BETA()
subject_list = list(range(1, 71))  # 被试编号从1到70
for s in subject_list:
    BETA_dataset.data_path(subject=s, path="E:\\MetaBCI-master\\mne_data")  # 依次为每个被试设置路径

events = BETA_dataset.events.keys()
freq_list = [str(BETA_dataset.get_freq(event)) for event in events]  # 获得所有刺激的频率
BETA_freq_map = {i: freq for i, freq in enumerate(freq_list)}  # 标签到频率的映射
print(freq_list)  # 输出频率显示

# ---------- Explicit experiment configuration ----------
t_start, t_end = 0.14, 1.14
sfreq = 250
window_length = t_end - t_start
n_samples = int(window_length * sfreq)

print(f"[BETA] Time window: {window_length:.2f}s ({n_samples} samples)")

# add 5-90Hz bandpass filter in raw hook
bandpass_low, bandpass_high = 5, 90
print(f"[BETA] Bandpass filter: {bandpass_low}-{bandpass_high} Hz")

def raw_hook(raw, caches):
    raw.filter(bandpass_low, bandpass_high, l_trans_bandwidth=2, h_trans_bandwidth=5, phase='zero-double')
    caches['raw_stage'] = caches.get('raw_stage', -1) + 1
    return raw, caches
    
# paradigm_2s用于S1-S16的被试(时间窗为2秒,提取1秒数据)
paradigm_2s = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],
    intervals=[(t_start, t_end)],  # 提取1秒数据
    events=freq_list,
    srate=sfreq
)

# paradigm_3s用于S17-S64的被试(时间窗为3秒,提取1秒数据)
paradigm_3s = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],
    intervals=[(t_start, t_end)],  # 提取1秒数据
    events=freq_list,
    srate=sfreq
)
paradigm_2s.register_raw_hook(raw_hook)
paradigm_3s.register_raw_hook(raw_hook)
#**************************************************
# 定义8组被试分组
#**************************************************
group_configs = [
    {'group_id': 1, 'subjects': list(range(1, 9)), 'paradigm': 'paradigm_2s', 'description': 'S1-S8 (2s paradigm)'},
    {'group_id': 2, 'subjects': list(range(9, 17)), 'paradigm': 'paradigm_2s', 'description': 'S9-S16 (2s paradigm)'},
    {'group_id': 3, 'subjects': list(range(17, 25)), 'paradigm': 'paradigm_3s', 'description': 'S17-S24 (3s paradigm)'},
    {'group_id': 4, 'subjects': list(range(25, 33)), 'paradigm': 'paradigm_3s', 'description': 'S25-S32 (3s paradigm)'},
    {'group_id': 5, 'subjects': list(range(33, 41)), 'paradigm': 'paradigm_3s', 'description': 'S33-S40 (3s paradigm)'},
]

--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchu

In [3]:
#**************************************************
# BETA数据集 EEGNET网络 - 分组LOSO实验
#**************************************************

# 设置device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 模型固定参数
F1 = 8
D = 2
F2 = 16
n_classes = 40
time_kernel_length = 64

# 文件储存在当前脚本所在的目录下
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

csv_filename = os.path.join(script_dir, "EEGnet_LOSO_grouped_experiment_BETA_1s.csv")

if os.path.exists(csv_filename):
    os.remove(csv_filename)
    print(f"🗑️ Removed existing file: {csv_filename}")

# CSV表头标志
header_written = False

# 困难样本挖掘Loss
class OhemCELoss(nn.Module):
    def __init__(self, thresh=0.7, n_min=16):
        super().__init__()
        self.thresh_prob = thresh
        self.n_min = n_min
        self.criteria = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, labels):
        loss = self.criteria(logits, labels)
        thresh = -torch.log(
            torch.tensor(self.thresh_prob, device=loss.device)
        )
        loss, _ = torch.sort(loss, descending=True)
        if loss[self.n_min - 1] > thresh:
            loss = loss[loss > thresh]
        else:
            loss = loss[: self.n_min]
        return loss.mean()

# 设置随机种子
set_random_seeds(42)

# 遍历每一组数据
for group_config in group_configs:
    group_id = group_config['group_id']
    subjects = group_config['subjects']
    paradigm_name = group_config['paradigm']
    description = group_config['description']
    
    print(f"\n{'#'*70}")
    print(f"# 🔬 Group {group_id}: {description}")
    print(f"# 被试编号: {subjects}")
    print(f"{'#'*70}")
    
    # 根据paradigm类型获取数据
    if paradigm_name == 'paradigm_2s':
        X_group, y_group, meta_group = paradigm_2s.get_data(
            BETA_dataset,
            subjects=subjects,
            return_concat=True,
            n_jobs=None,
            verbose=False
        )
    else:  # paradigm_3s
        X_group, y_group, meta_group = paradigm_3s.get_data(
            BETA_dataset,
            subjects=subjects,
            return_concat=True,
            n_jobs=None,
            verbose=False
        )
    
    # 获取数据维度
    n_channels = X_group.shape[1]
    n_samples = X_group.shape[2]
    
    # 获取当前组的所有被试ID
    all_subjects = np.unique(meta_group['subject'])
    N = len(all_subjects)
    print(f"Total subjects in group: {N}")
    print(f"Subject IDs: {all_subjects}")
    print(f"Data shape: {X_group.shape}")
    
    # 存储当前组每个被试的准确率
    acc_list = []
    loso_results = []
    
    # LOSO循环：每个被试作为测试集
    for i, test_subject in enumerate(all_subjects):
        print(f"\n{'='*50}")
        print(f"🧪 Group {group_id} - Fold {i+1}/{N}: Testing on Subject {test_subject}")
        print(f"{'='*50}")
        
        # LOSO划分
        test_mask = meta_group['subject'] == test_subject
        train_mask = ~test_mask
        
        train_ind = np.where(train_mask)[0]
        test_ind = np.where(test_mask)[0]
        
        print(f"Training samples: {len(train_ind)}")
        print(f"Testing samples: {len(test_ind)}")
        # 初始化模型
        estimator = EEGNet(
            F1, D, F2, n_channels, n_samples, n_classes,
            time_kernel_length
        )
        estimator.set_params(
                optimizer__lr=1e-3,
                device=device,
                #criterion=nn.CrossEntropyLoss,
                criterion=OhemCELoss,
                criterion__thresh=0.7,
                criterion__n_min=16,
            )
        
        # 验证集 = 测试集
        valid_ds = torch.utils.data.TensorDataset(
            torch.tensor(X_group[test_ind], dtype=torch.float64),
            torch.tensor(y_group[test_ind], dtype=torch.long)
        )
        estimator.set_params(train_split=predefined_split(valid_ds))
        
        # 训练模型
        estimator.fit(X_group[train_ind], y_group[train_ind])
        
        # 测试集评估
        p_labels = estimator.predict(X_group[test_ind])
        y_test = y_group[test_ind]
        
        # 计算当前被试的准确率
        acc_s = np.mean(p_labels == y_test)
        acc_list.append(acc_s)
        
        print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%)")
        
        # 保存当前被试结果
        loso_results.append({
            'group_id': group_id,
            'group_description': description,
            'row_type': 'subject',
            'test_subject': test_subject,
            'accuracy': acc_s,
            'n_test_samples': len(test_ind),
            'n_correct': int(np.sum(p_labels == y_test)),
            'mean_accuracy': '',
            'std_accuracy': ''
        })
    
    # 当前组的汇总统计
    acc_array = np.array(acc_list)
    mean_acc = acc_array.mean()
    std_acc = acc_array.std()
    
    print(f"\n{'='*60}")
    print(f"📊 Summary for Group {group_id}")
    print(f"   Mean ± Std: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
    print(f"{'='*60}")
    
    # 添加summary行
    loso_results.append({
        'group_id': group_id,
        'group_description': description,
        'row_type': 'summary',
        'test_subject': 'ALL',
        'accuracy': '',
        'n_test_samples': N,
        'n_correct': '',
        'mean_accuracy': mean_acc,
        'std_accuracy': std_acc
    })
    
    # 追加写入CSV
    results_df = pd.DataFrame(loso_results)
    
    if not header_written:
        results_df.to_csv(csv_filename, mode='w', index=False, header=True)
        header_written = True
    else:
        results_df.to_csv(csv_filename, mode='a', index=False, header=False)
    
    print(f"💾 Results for Group {group_id} appended to: {csv_filename}")

# ========== 实验完成 ==========
print(f"\n{'#'*70}")
print(f"# 🎉 All experiments completed!")
print(f"# 📁 Results saved to: {csv_filename}")
print(f"{'#'*70}")

# 读取并显示汇总结果
final_df = pd.read_csv(csv_filename)
summary_df = final_df[final_df['row_type'] == 'summary'][['group_id', 'group_description', 'mean_accuracy', 'std_accuracy']]
print("\n📋 Summary Table:")
print(summary_df.to_string(index=False))

Device: cuda
🗑️ Removed existing file: e:\MetaBCI-master\EEGnet_LOSO_grouped_experiment_BETA_1s.csv

######################################################################
# 🔬 Group 1: S1-S8 (2s paradigm)
# 被试编号: [1, 2, 3, 4, 5, 6, 7, 8]
######################################################################
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
Total subjects in group: 8
Subject IDs: [1 2 3 4 5 6 7 8]
Data shape: (1280, 9, 250)

🧪 Group 1 - Fold 1/8: Testing on Subject 1
Training samples: 1120
Testing samples: 160
  epoch    train_acc    train_loss    

In [4]:
#**************************************************
# Benchmark数据集读取处理
#**************************************************
Bench_dataset = Wang2016()
# 扩展到32个被试
Bench_subject_list = list(range(1, 33))

for s in Bench_subject_list:
    Bench_dataset.data_path(subject=s, path="E:\\MetaBCI-master\\mne_data")

events = Bench_dataset.events.keys()
freq_list = [str(Bench_dataset.get_freq(event)) for event in events]  # 获得所有刺激的频率

# ---------- Explicit experiment configuration ----------
t_start, t_end = 0.14, 1.14
sfreq = 250
window_length = t_end - t_start
n_samples = int(window_length * sfreq)

print(f"[Benchmark] Time window: {window_length:.2f}s ({n_samples} samples)")

# add 5-90Hz bandpass filter in raw hook
bandpass_low, bandpass_high = 5, 90
print(f"[Benchmark] Bandpass filter: {bandpass_low}-{bandpass_high} Hz")

def raw_hook(raw, caches):
    raw.filter(bandpass_low, bandpass_high, l_trans_bandwidth=2, h_trans_bandwidth=5, phase='zero-double')
    caches['raw_stage'] = caches.get('raw_stage', -1) + 1
    return raw, caches

Bench_paradigm = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],  # 选择电极通道
    intervals=[(t_start, t_end)],
    events=freq_list,
    srate=sfreq
)

Bench_paradigm.register_raw_hook(raw_hook)

#**************************************************
# 定义4组被试分组
#**************************************************
group_configs = [
    {'group_id': 1, 'subjects': list(range(1, 9)), 'description': 'S1-S8'},
    {'group_id': 2, 'subjects': list(range(9, 17)), 'description': 'S9-S16'},
    {'group_id': 3, 'subjects': list(range(17, 25)), 'description': 'S17-S24'},
    {'group_id': 4, 'subjects': list(range(25, 33)), 'description': 'S25-S32'}
]

--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yijun/S23.mat.7z
--------ssssss, /upload/yijun/S24.mat.7z
--------ssssss, /upload/y

In [5]:
#**************************************************
# Benchmark数据集 EEGNET网络 - 分组LOSO实验
#**************************************************

# 设置device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 模型固定参数
F1 = 8
D = 2
F2 = 16
n_classes = 40
time_kernel_length = 64

# 文件储存在当前脚本所在的目录下
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

csv_filename = os.path.join(script_dir, "EEGnet_LOSO_grouped_experiment_Benchmark_1s.csv")

if os.path.exists(csv_filename):
    os.remove(csv_filename)
    print(f"🗑️ Removed existing file: {csv_filename}")

# CSV表头标志
header_written = False

# 困难样本挖掘Loss
class OhemCELoss(nn.Module):
    def __init__(self, thresh=0.7, n_min=16):
        super().__init__()
        self.thresh_prob = thresh
        self.n_min = n_min
        self.criteria = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, labels):
        loss = self.criteria(logits, labels)
        thresh = -torch.log(
            torch.tensor(self.thresh_prob, device=loss.device)
        )
        loss, _ = torch.sort(loss, descending=True)
        if loss[self.n_min - 1] > thresh:
            loss = loss[loss > thresh]
        else:
            loss = loss[: self.n_min]
        return loss.mean()

# 设置随机种子
set_random_seeds(42)

# 遍历每一组数据
for group_config in group_configs:
    group_id = group_config['group_id']
    subjects = group_config['subjects']
    description = group_config['description']
    
    print(f"\n{'#'*70}")
    print(f"# 🔬 Group {group_id}: {description}")
    print(f"# 被试编号: {subjects}")
    print(f"{'#'*70}")
    
    # 获取当前组的数据
    print(f"[Benchmark] Loading data for subjects: {subjects}")
    X_group, y_group, meta_group = Bench_paradigm.get_data(
        Bench_dataset,
        subjects=subjects,
        return_concat=True,
        n_jobs=None,
        verbose=False
    )
    
    # 获取数据维度
    n_channels = X_group.shape[1]
    n_samples = X_group.shape[2]
    
    # 获取当前组的所有被试ID
    all_subjects = np.unique(meta_group['subject'])
    N = len(all_subjects)
    print(f"Total subjects in group: {N}")
    print(f"Subject IDs: {all_subjects}")
    print(f"Data shape: {X_group.shape}")
    
    # 存储当前组每个被试的准确率
    acc_list = []
    loso_results = []
    
    # LOSO循环：每个被试作为测试集
    for i, test_subject in enumerate(all_subjects):
        print(f"\n{'='*50}")
        print(f"🧪 Group {group_id} - Fold {i+1}/{N}: Testing on Subject {test_subject}")
        print(f"{'='*50}")
        
        # LOSO划分
        test_mask = meta_group['subject'] == test_subject
        train_mask = ~test_mask
        
        train_ind = np.where(train_mask)[0]
        test_ind = np.where(test_mask)[0]
        
        print(f"Training samples: {len(train_ind)}")
        print(f"Testing samples: {len(test_ind)}")

        
        # 初始化模型
        estimator = EEGNet(
            F1, D, F2, n_channels, n_samples, n_classes,
            time_kernel_length
        )
        estimator.set_params(
                optimizer__lr=1e-3,
                device=device,
                #criterion=nn.CrossEntropyLoss,
                criterion=OhemCELoss,
                criterion__thresh=0.7,
                criterion__n_min=16,
            )
        
        # 验证集 = 测试集
        valid_ds = torch.utils.data.TensorDataset(
            torch.tensor(X_group[test_ind], dtype=torch.float64),
            torch.tensor(y_group[test_ind], dtype=torch.long)
        )
        estimator.set_params(train_split=predefined_split(valid_ds))
        
        # 训练模型
        estimator.fit(X_group[train_ind], y_group[train_ind])
        
        # 测试集评估
        p_labels = estimator.predict(X_group[test_ind])
        y_test = y_group[test_ind]
        
        # 计算当前被试的准确率
        acc_s = np.mean(p_labels == y_test)
        acc_list.append(acc_s)
        
        print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%)")
        
        # 保存当前被试结果
        loso_results.append({
            'group_id': group_id,
            'group_description': description,
            'row_type': 'subject',
            'test_subject': test_subject,
            'accuracy': acc_s,
            'n_test_samples': len(test_ind),
            'n_correct': int(np.sum(p_labels == y_test)),
            'mean_accuracy': '',
            'std_accuracy': ''
        })
    
    # 当前组的汇总统计
    acc_array = np.array(acc_list)
    mean_acc = acc_array.mean()
    std_acc = acc_array.std()
    
    print(f"\n{'='*60}")
    print(f"📊 Summary for Group {group_id}")
    print(f"   Mean ± Std: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
    print(f"{'='*60}")
    
    # 添加summary行
    loso_results.append({
        'group_id': group_id,
        'group_description': description,
        'row_type': 'summary',
        'test_subject': 'ALL',
        'accuracy': '',
        'n_test_samples': N,
        'n_correct': '',
        'mean_accuracy': mean_acc,
        'std_accuracy': std_acc
    })
    
    # 追加写入CSV
    results_df = pd.DataFrame(loso_results)
    
    if not header_written:
        results_df.to_csv(csv_filename, mode='w', index=False, header=True)
        header_written = True
    else:
        results_df.to_csv(csv_filename, mode='a', index=False, header=False)
    
    print(f"💾 Results for Group {group_id} appended to: {csv_filename}")

# ========== 实验完成 ==========
print(f"\n{'#'*70}")
print(f"# 🎉 All experiments completed!")
print(f"# 📁 Results saved to: {csv_filename}")
print(f"{'#'*70}")

# 读取并显示汇总结果
final_df = pd.read_csv(csv_filename)
summary_df = final_df[final_df['row_type'] == 'summary'][['group_id', 'group_description', 'mean_accuracy', 'std_accuracy']]
print("\n📋 Summary Table:")
print(summary_df.to_string(index=False))


Device: cuda
🗑️ Removed existing file: e:\MetaBCI-master\EEGnet_LOSO_grouped_experiment_Benchmark_1s.csv

######################################################################
# 🔬 Group 1: S1-S8
# 被试编号: [1, 2, 3, 4, 5, 6, 7, 8]
######################################################################
[Benchmark] Loading data for subjects: [1, 2, 3, 4, 5, 6, 7, 8]
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
Total subjects in group: 8
Subject IDs: [1 2 3 4 5 6 7 8]
Data shape: (1920, 9, 250)

🧪 Group 1 - Fold 1/8: Testing on Subject 1
Training samples: 1680
Testing samples: 240
  epoch    train_acc    train_loss    valid_acc    valid_loss    cp     dur
-------  -----------  ------------  -----------  ------------  ----